In [1]:
import duckdb
import pandas as pd
import numpy as np
import json
import yaml
import requests
import random
import re

from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

con = duckdb.connect()

In [2]:
CKAN_API = "https://data.gov.au/data/api/3/action"

dataset_id = "7b8656f9-606d-4337-af29-66b89b2eeefb"  # ASIC - Company Dataset

ds = requests.get(
    f"{CKAN_API}/package_show",
    params={"id": dataset_id},
    timeout=30
).json()["result"]

print("Title:", ds["title"])
print("Licence:", ds.get("license_title"))

resources = pd.DataFrame([
    {
        "id": r.get("id"),
        "name": r.get("name"),
        "format": r.get("format"),
        "url": r.get("url"),
        "size": r.get("size"),
    }
    for r in ds.get("resources", [])
])

resources

Title: ASIC - Company Dataset
Licence: Creative Commons Attribution 3.0 Australia


,id,name,format,url,size
0,9430b6b2-b28c-4b1a-a58a-a88b54912853,Company Dataset - Help File,PDF,https://data.gov.au/data/dataset/7b8656f9-606d-4337-af29-66b89b2eeefb/resource/9430b6b2-b28c-4b1a-a58a-a88b54912853/...,466372
1,5c3914e6-413e-4a2c-b890-bf8efe3eabf2,Company Dataset - Current,CSV,https://data.gov.au/data/dataset/7b8656f9-606d-4337-af29-66b89b2eeefb/resource/5c3914e6-413e-4a2c-b890-bf8efe3eabf2/...,400098721
2,d6d03876-71a4-4e82-8c77-2e4df5da4236,Company Dataset - Current,ZIP,https://data.gov.au/data/dataset/7b8656f9-606d-4337-af29-66b89b2eeefb/resource/d6d03876-71a4-4e82-8c77-2e4df5da4236/...,79151729


In [3]:
resources[["name", "format", "url"]]

,name,format,url
0,Company Dataset - Help File,PDF,https://data.gov.au/data/dataset/7b8656f9-606d-4337-af29-66b89b2eeefb/resource/9430b6b2-b28c-4b1a-a58a-a88b54912853/...
1,Company Dataset - Current,CSV,https://data.gov.au/data/dataset/7b8656f9-606d-4337-af29-66b89b2eeefb/resource/5c3914e6-413e-4a2c-b890-bf8efe3eabf2/...
2,Company Dataset - Current,ZIP,https://data.gov.au/data/dataset/7b8656f9-606d-4337-af29-66b89b2eeefb/resource/d6d03876-71a4-4e82-8c77-2e4df5da4236/...


In [4]:
resource = next(
    r for r in ds["resources"]
    if (r.get("format") or "").upper() in {"CSV", "XLSX", "XLS"}
)

resource

{'cache_last_updated': None,
 'cache_url': None,
 'ckan_url': 'https://data.gov.au',
 'created': '2022-02-23T00:17:52.185051',
 'datastore_active': True,
 'datastore_contains_all_records_of_source_file': True,
 'description': 'Company Dataset extract as at 29/09/2026 00:50',
 'format': 'CSV',
 'hash': 'c4700f080ae9d3548ac1800d52ff950c',
 'id': '5c3914e6-413e-4a2c-b890-bf8efe3eabf2',
 'ignore_hash': False,
 'last_modified': '2026-09-28T14:51:25.555707',
 'last_modified ': '29/09/2026 00:50',
 'metadata_modified': '2026-09-28T15:24:24.332244',
 'mimetype': 'text/csv',
 'mimetype_inner': None,
 'name': 'Company Dataset - Current',
 'original_url': 'https://data.gov.au/data/dataset/7b8656f9-606d-4337-af29-66b89b2eeefb/resource/5c3914e6-413e-4a2c-b890-bf8efe3eabf2/download/company_202609.csv',
 'package_id': '7b8656f9-606d-4337-af29-66b89b2eeefb',
 'position': 1,
 'resource_id': '5c3914e6-413e-4a2c-b890-bf8efe3eabf2',
 'resource_type': None,
 'set_url_type': False,
 'size': 400098721,
 'sta

In [5]:
url = resource["url"]

con.execute(f"""
    CREATE OR REPLACE VIEW source_data AS
    SELECT *
    FROM read_csv_auto('{url}', sample_size=-1)
""")

In [7]:
con.execute("""
    SELECT *
    FROM source_data
    LIMIT 10
""").df()

,Company Name,ACN,Type,Class,Sub Class,Status,Date of Registration,Date of Deregistration,Previous State of Registration,State Registration number,Modified since last report,Current Name Indicator,ABN,Current Name,Current Name Start Date
0,LOVINI HOLDINGS PTY LTD,000000019,APTY,LMSH,PROP,REGD,1990-01-08,NaT,NSW,46869041,None,NaN,89000000019,MONAKA PTY LTD,2016-01-28
1,MONAKA PTY LTD,000000019,APTY,LMSH,PROP,REGD,1990-01-08,NaT,NSW,46869041,None,Y,89000000019,NaN,NaT
2,CHRISTENSEN & ASSOCIATES PTY LTD,000000028,APTY,LMSH,PSTC,REGD,1987-09-15,NaT,NSW,40398926,None,Y,91000000028,NaN,NaT
3,TASDARK PTY LTD,000000028,APTY,LMSH,PSTC,REGD,1987-09-15,NaT,NSW,40398926,None,NaN,91000000028,CHRISTENSEN & ASSOCIATES PTY LTD,2007-09-24
4,DHATA HOLDINGS PTY. LTD.,000000046,APTY,LMSH,PSTC,REGD,1988-03-17,NaT,NSW,41493048,None,Y,95000000046,NaN,NaT
5,PATIA HOLDINGS PTY. LTD.,000000055,APTY,LMSH,PROP,REGD,1989-10-09,NaT,NSW,46389303,None,Y,97000000055,NaN,NaT
6,SAMOZI PTY LTD,000000082,APTY,LMSH,PROP,REGD,1984-02-13,NaT,NSW,33121533,None,NaN,14000000082,W & L COMPUTER CONSULTANTS PTY LTD,1984-03-28
7,W & L COMPUTER CONSULTANTS PTY LTD,000000082,APTY,LMSH,PROP,REGD,1984-02-13,NaT,NSW,33121533,None,Y,14000000082,NaN,NaT
8,BANQUE NATIONALE DE PARIS,000000117,FNOS,NONE,NONE,REGD,1907-04-30,NaT,NSW,00002239,None,NaN,23000000117,BNP PARIBAS,2000-06-02
9,BNP PARIBAS,000000117,FNOS,NONE,NONE,REGD,1907-04-30,NaT,NSW,00002239,None,Y,23000000117,NaN,NaT


In [12]:
from pathlib import Path
import requests
import re



def safe_name(text):
    text = text.strip()
    text = re.sub(r"[^\w\s-]", "", text)     # remove punctuation
    text = re.sub(r"[-\s]+", "_", text)      # spaces/hyphens -> _
    return text.strip("_")


def download_resource(ds, resource, root="data"):
    dataset_name = safe_name(ds["title"])
    resource_name = safe_name(resource.get("name", "resource"))

    fmt = (resource.get("format") or "").lower().strip(".") or "data"

    dataset_folder = Path(root) / dataset_name
    dataset_folder.mkdir(parents=True, exist_ok=True)

    local_path = dataset_folder / f"{resource_name}.{fmt}"

    if local_path.exists():
        print("Already exists:", local_path)
        return str(local_path)

    with requests.get(resource["url"], stream=True, timeout=120) as r:
        r.raise_for_status()

        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    print("Saved:", local_path)
    return str(local_path)

In [13]:
resource = next(
    r for r in ds["resources"]
    if (r.get("format") or "").upper() == "CSV"
)

local_path = download_resource(ds, resource)

Saved: data/ASIC_Company_Dataset/Company_Dataset_Current.csv


In [15]:
local_path

'data/ASIC_Company_Dataset/Company_Dataset_Current.csv'

In [17]:
con.execute("DROP VIEW IF EXISTS source_data")
con.execute("DROP TABLE IF EXISTS source_data")

con.execute(f"""
    CREATE TABLE source_data AS
    SELECT *
    FROM read_csv_auto(
        '{local_path}',
        sample_size=-1
    )
""")

In [18]:
con.execute("""
    SELECT COUNT(*) AS row_count
    FROM source_data
""").df()

,row_count
0,4448915


In [19]:
schema = con.execute("""
    DESCRIBE source_data
""").df()

schema

,column_name,column_type,null,key,default,extra
0,Company Name,VARCHAR,YES,None,None,None
1,ACN,VARCHAR,YES,None,None,None
2,Type,VARCHAR,YES,None,None,None
3,Class,VARCHAR,YES,None,None,None
4,Sub Class,VARCHAR,YES,None,None,None
5,Status,VARCHAR,YES,None,None,None
6,Date of Registration,DATE,YES,None,None,None
7,Date of Deregistration,DATE,YES,None,None,None
8,Previous State of Registration,VARCHAR,YES,None,None,None
9,State Registration number,VARCHAR,YES,None,None,None


In [20]:
row_count = con.execute("""
    SELECT COUNT(*) FROM source_data
""").fetchone()[0]

column_count = len(schema)

print("Rows:", row_count)
print("Columns:", column_count)

Rows: 4448915
Columns: 15


In [36]:
import re
import json


class DatasetProfiler:

    def __str__(self):
        print(f"DatasetProfiler for table: {self.table}")
        
    def __init__(
        self,
        con,
        table="source_data",
        metadata=None,
        max_text_len=100,
        top_n_categories=20
    ):
        self.con = con
        self.table = table
        self.metadata = metadata or {}
        self.max_text_len = max_text_len
        self.top_n_categories = top_n_categories

    def _q(self, name):
        """Safely quote DuckDB identifiers."""
        return '"' + str(name).replace('"', '""') + '"'

    def _truncate(self, value):
        if value is None:
            return None

        value = str(value)

        if len(value) > self.max_text_len:
            return value[:self.max_text_len] + "..."

        return value

    def _truncate_records(self, df):
        records = df.to_dict("records")

        return [
            {
                k: self._truncate(v)
                for k, v in row.items()
            }
            for row in records
        ]

    def _semantic_hints(self, name, col_type, stats):
        """
        Cheap deterministic hints.
        These are evidence for the agent, NOT final mappings.
        """
        name_lower = name.lower()

        hints = []

        # Column-name hints
        if any(x in name_lower for x in [
            "id", "identifier", "number", "abn", "acn", "nzbn"
        ]):
            hints.append("possible_identifier")

        if "abn" in name_lower:
            hints.append("possible_abn")

        if "acn" in name_lower:
            hints.append("possible_acn")

        if any(x in name_lower for x in [
            "date", "created", "registered", "updated", "from", "to"
        ]):
            hints.append("possible_date")

        if any(x in name_lower for x in [
            "status", "type", "state", "category",
            "class", "industry", "code"
        ]):
            hints.append("possible_enum_or_category")

        if any(x in name_lower for x in [
            "name", "organisation", "organization",
            "company", "business", "trading"
        ]):
            hints.append("possible_name")

        if any(x in name_lower for x in [
            "address", "street", "locality", "suburb", "postcode"
        ]):
            hints.append("possible_address")

        if "url" in name_lower or "website" in name_lower:
            hints.append("possible_url")

        # Data hints
        uniqueness = stats.get("uniqueness_ratio")

        if uniqueness is not None and uniqueness > 0.95:
            hints.append("high_uniqueness")

        distinct = stats.get("distinct_count")

        if distinct is not None and distinct <= 50:
            hints.append("low_cardinality")

        return list(dict.fromkeys(hints))

    def _column_profile(self, column, col_type, row_count):
        c = self._q(column)

        # Basic stats
        result = self.con.execute(f"""
            SELECT
                COUNT(*) FILTER (WHERE {c} IS NULL) AS null_count,
                COUNT(DISTINCT {c}) AS distinct_count
            FROM {self._q(self.table)}
        """).fetchone()

        null_count, distinct_count = result

        non_null_count = row_count - null_count

        null_ratio = (
            null_count / row_count
            if row_count else 0
        )

        uniqueness_ratio = (
            distinct_count / non_null_count
            if non_null_count else 0
        )

        stats = {
            "null_count": null_count,
            "null_ratio": round(null_ratio, 4),
            "distinct_count": distinct_count,
            "uniqueness_ratio": round(uniqueness_ratio, 4),
        }

        # String-length stats
        length_stats = self.con.execute(f"""
            SELECT
                MIN(LENGTH(CAST({c} AS VARCHAR))),
                MAX(LENGTH(CAST({c} AS VARCHAR))),
                AVG(LENGTH(CAST({c} AS VARCHAR)))
            FROM {self._q(self.table)}
            WHERE {c} IS NOT NULL
        """).fetchone()

        stats["length"] = {
            "min": length_stats[0],
            "max": length_stats[1],
            "avg": round(length_stats[2], 2)
                if length_stats[2] is not None else None
        }

        # Examples
        examples = self.con.execute(f"""
            SELECT DISTINCT {c}
            FROM {self._q(self.table)}
            WHERE {c} IS NOT NULL
            LIMIT 5
        """).fetchall()

        stats["examples"] = [
            self._truncate(x[0])
            for x in examples
        ]

        # Top categorical values
        # Particularly useful for status/type/state/etc.
        if distinct_count <= self.top_n_categories or any(
            x in column.lower()
            for x in [
                "status", "type", "state", "category",
                "class", "industry", "code"
            ]
        ):
            top_values = self.con.execute(f"""
                SELECT
                    {c} AS value,
                    COUNT(*) AS count
                FROM {self._q(self.table)}
                WHERE {c} IS NOT NULL
                GROUP BY {c}
                ORDER BY count DESC
                LIMIT {self.top_n_categories}
            """).fetchall()

            stats["top_values"] = [
                {
                    "value": self._truncate(value),
                    "count": count
                }
                for value, count in top_values
            ]

        stats["hints"] = self._semantic_hints(
            column,
            col_type,
            stats
        )

        return stats

    def profile(self):
        table = self._q(self.table)

        # Dataset-level size
        row_count = self.con.execute(
            f"SELECT COUNT(*) FROM {table}"
        ).fetchone()[0]

        # Schema
        schema_df = self.con.execute(
            f"DESCRIBE {table}"
        ).df()

        schema = [
            {
                "name": row["column_name"],
                "type": row["column_type"]
            }
            for _, row in schema_df.iterrows()
        ]

        # Per-column profiling
        columns = {}

        for col in schema:
            print("Profiling:", col["name"])

            columns[col["name"]] = {
                "duckdb_type": col["type"],
                **self._column_profile(
                    col["name"],
                    col["type"],
                    row_count
                )
            }

        # -----------------------
        # Record samples
        # -----------------------

        head = self.con.execute(f"""
            SELECT *
            FROM {table}
            LIMIT 3
        """).df()

        random_rows = self.con.execute(f"""
            SELECT *
            FROM {table}
            USING SAMPLE 5 ROWS
        """).df()

        tail_offset = max(row_count - 2, 0)

        tail = self.con.execute(f"""
            SELECT *
            FROM {table}
            OFFSET {tail_offset}
            LIMIT 2
        """).df()

        return {
            "metadata": self.metadata,

            "dataset_stats": {
                "row_count": row_count,
                "column_count": len(schema),
            },

            "schema": schema,

            "columns": columns,

            "samples": {
                "head": self._truncate_records(head),
                "random": self._truncate_records(random_rows),
                "tail": self._truncate_records(tail),
            }
        }

    def to_json(self, profile):
        return json.dumps(
            profile,
            indent=2,
            default=str
        )

In [37]:
metadata = {
    "dataset_id": ds.get("id"),
    "title": ds.get("title"),
    "description": ds.get("notes"),
    "publisher": (ds.get("organization") or {}).get("title"),
    "licence": ds.get("license_title"),
    "resource_id": resource.get("id"),
    "resource_name": resource.get("name"),
    "resource_format": resource.get("format"),
    "resource_url": resource.get("url"),
}

profiler = DatasetProfiler(
    con,
    table="source_data",
    metadata=metadata
)

profile = profiler.profile()

Profiling: Company Name
Profiling: ACN
Profiling: Type
Profiling: Class
Profiling: Sub Class
Profiling: Status
Profiling: Date of Registration
Profiling: Date of Deregistration
Profiling: Previous State of Registration
Profiling: State Registration number
Profiling: Modified since last report
Profiling: Current Name Indicator
Profiling: ABN
Profiling: Current Name
Profiling: Current Name Start Date


In [30]:
import pprint

In [32]:
pprint.pprint(profiler.to_json(profile))

('{\n'
 '  "metadata": {\n'
 '    "dataset_id": "7b8656f9-606d-4337-af29-66b89b2eeefb",\n'
 '    "title": "ASIC - Company Dataset",\n'
 '    "description": "###Update March 2025 ###\\r\\n\\r\\nFrom 11 March 2025, '
 'the dataset will be updated to include 1 new field, Date of Deregistration, '
 '(see help file for details). \\r\\n\\r\\n###Update August 2018 - frequency '
 'change to Company dataset ###\\r\\n\\r\\nFrom 7 August 2018, the Company '
 'dataset will be updated weekly every Tuesday. As a result, the information '
 'might not be accurate at the time you check the Company '
 'dataset.\\r\\n[ASIC-Connect](https://connectonline.asic.gov.au/ '
 '\\"ASIC-Connect\\") updates information in real time, therefore, please '
 'consider accessing information on that platform if you need up to date '
 'information.\\r\\n\\r\\n***\\r\\n\\r\\n###Dataset summary###\\r\\nASIC is '
 'Australia\\u2019s corporate, markets and financial services regulator. ASIC '
 'contributes to Australia\\u2019

In [38]:
from pathlib import Path
import json

profile_path = Path(
    "data/ASIC_Company_Dataset/Company_Dataset_Current.profile.json"
)

profile_path.parent.mkdir(parents=True, exist_ok=True)

with open(profile_path, "w", encoding="utf-8") as f:
    json.dump(profile, f, indent=2, ensure_ascii=False, default=str)

print("Saved:", profile_path)

Saved: data/ASIC_Company_Dataset/Company_Dataset_Current.profile.json
